In [ ]:
import os
import pandas as pd

In [ ]:
def folder_init(folder_name):
    if not os.path.exists(folder_name):
        os.mkdir(folder_name)

In [ ]:
def write_dfs(oscar_df, imdb_df):
    data_folder = 'filtered/'
    folder_init(data_folder)
    oscar_df.to_csv(data_folder + 'oscar_filtered.csv')
    imdb_df.to_csv(data_folder + 'imdb_filtered.csv')

In [ ]:
def write_joined(oscar_with_imdb, imdb_non_oscar):
    data_folder = 'filtered/'
    folder_init(data_folder)
    oscar_with_imdb.to_csv(data_folder +  'oscar_with_imdb.csv')
    imdb_non_oscar.to_csv(data_folder + 'imdb_non_oscar.csv')

In [ ]:
def read_data(file1, file2):
    folder_name = 'filtered/'
    return (pd.read_csv(folder_name + file1), pd.read_csv(folder_name + file2))

In [ ]:
def convert_num(num, pos):
    # Format with K/M/B for thousands/millions/billions
    if num >= 1_000_000_000:
        return f'{num/1_000_000_000:.1f}B'
    elif num >= 1_000_000:
        return f'{num/1_000_000:.1f}M'
    elif num >= 1_000:
        return f'{num/1_000:.1f}K'
    else:
        return str(int(num))


In [ ]:
# Used for budget and box office fields
def round_value(x):
    if x is None or pd.isna(x):
        return None  # Keep as missing so dropping NAs later works
    else:
        x = float(x)
        
    if x <= 0:    # Negative or zero budget/gross, probably bad data
        return None
    elif x > 100_000:
        return round(x, -3)  # nearest thousand
    elif x > 10_000_000:
        return round(x, -5)  # nearest hundred thousand
    elif x > 100_000_000:
        return round(x, -7)  # nearest ten million

In [ ]:
import datetime

def convert_date(release_date):
    if isinstance(release_date, datetime.datetime):
        return release_date

    if isinstance(release_date, int):
        return datetime(release_date, 1, 1)

    try:
        return datetime.datetime.fromisoformat(release_date)
    except ValueError:
        raise ValueError(f'String {release_date!r} is not in valid datetime format')
    

In [ ]:
# count = 0
# conversion_map = {}

In [ ]:
from price_parser import Price
import numpy as np

def normalize_budget(budget, release_date, converter):
    currencies = {
        '$': 'USD',
        '£': 'GBP',
        '¥': 'YEN',
        '₩': 'KRW',
        '£': 'GBP',
        '¥': 'JPY',
        '₩': 'KRW',
        '₪': 'ILS',
        '₱': 'PHP',
        '₹': 'INR',
        'CA$': 'CAD',
        'CN¥': 'CNY',
        '€': 'EUR',
        'R$': 'BRL',
        'NZ$': 'NZD',
        'NT$': 'TWD',
        'MX$': 'MXN',
        'HK$': 'HKD',
        'CA$': 'CAD',
        'A$': 'AUD'
    }
    
    if type(budget) == float:
        return budget
    # 1. Parse price
    if type(budget) == str:
        space_idx = budget.find(' ')
        budget = budget[:space_idx]
        
    parsed = Price.fromstring(budget)
    amount = parsed.amount_float
    currency = parsed.currency
    
    if not currency:
        print('Could not determine')
        print(budget, parsed, amount, '\n')
        return 0
    
    if currency in currencies:
        currency = currencies[currency]

    # print(parsed, amount, currency)
    # 2. Convert to USD using release year
    historical_amount = get_historical_rate(int(amount), currency, 'USD', release_date, converter)

    # 3. Adjust for inflation
    # adjusted_amount = adjust_for_inflation(usd_amount, row['release_year'])
    # print(adjusted_amount, 'Good')

    return historical_amount

# all_films_scatter['adjusted_budget'] = all_films_scatter.apply(normalize_budget, axis=1)


In [ ]:
# Might have to skip this
# def adjust_for_inflation()

In [ ]:
from currency_converter import CurrencyConverter
import datetime

def get_historical_rate(amount, from_currency, to_currency, release_date, converter):
    try:
        date_time_obj = convert_date(release_date)
        
        # if conversion in conversion_map:
        #     rate = conversion_map[conversion]
        # else:
        rate = converter.convert(amount, from_currency, to_currency, date=date_time_obj)
        return rate
    except Exception as e:
        # print(rate)
        print(f"Error getting historical rate for {from_currency}->{to_currency} in {release_date}: {e}\n")
        return 0
